# Results

Map `DocumentAnswers` to the same columns the UI shows, without Streamlit.
The first code cell uses an in-memory sample. Later cells call the pipeline.
Replace the question template before the setup cell. The last cell deletes this collection.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

from helpers import answered_counts, result_rows

from models.query import (
    AgentAnswer,
    AnswerConfidence,
    AnswerSource,
    Citation,
    DocumentAnswers,
    QuestionType,
    UserQuestion,
)

field = UserQuestion(
    label="Field",
    question="Copy the requested value from the document.",
    question_type=QuestionType.EXTRACTION,
)
other = UserQuestion(
    label="Other field",
    question="Copy the other requested value from the document.",
    question_type=QuestionType.EXTRACTION,
)
sample = [
    DocumentAnswers(
        document="sample.pdf",
        answers=[
            AgentAnswer(
                question=field,
                answer="sample",
                citation=Citation(
                    text="sample fragment",
                    source="sample.pdf",
                    page=1,
                    score=0.9,
                ),
                document="sample.pdf",
                confidence=AnswerConfidence.HIGH,
                answer_source=AnswerSource.DIRECT,
            ),
            AgentAnswer(
                question=other,
                document="sample.pdf",
                confidence=AnswerConfidence.NOT_FOUND,
            ),
        ],
    )
]
for row in result_rows(sample):
    print(row)
for count in answered_counts(sample):
    print(count)

In [ ]:
from helpers import accepted_pdfs, new_session_id, timed

from pipeline.embedder import embed_document
from pipeline.extractor import extract
from pipeline.rag_agent import answer_question

pdf_dir = (
    Path(os.environ["NOTEBOOK_PDF_DIR"])
    if os.environ.get("NOTEBOOK_PDF_DIR")
    else ROOT / "notebooks" / "inputs"
)
questions = [
    UserQuestion(
        label="Field",
        question="Copy the requested value from the document.",
        question_type=QuestionType.EXTRACTION,
    )
]
session_id = new_session_id()
collected = []

with timed("setup") as setup:
    accepted, rejected = accepted_pdfs(pdf_dir)
    for path in accepted:
        try:
            document = await extract(path)
        except Exception as exc:
            print(path.name, "failure", type(exc).__name__, exc)
            continue
        if document.metadata.total_chunks == 0:
            print(path.name, "failure", "RuntimeError", "produced no text")
            continue
        await embed_document(document, session_id)
        filename = document.metadata.filename
        answers = [
            answer_question(question, session_id, filename) for question in questions
        ]
        collected.append((filename, answers))

print(session_id)
print(f"setup: {setup['seconds']:.3f}s")
print(f"ready: {len(collected)}  rejected: {len(rejected)}")

In [ ]:
with timed("map") as mapping:
    results = [
        DocumentAnswers(document=filename, answers=answers)
        for filename, answers in collected
    ]
    rows = result_rows(results)
    counts = answered_counts(results)

print(f"map: {mapping['seconds']:.3f}s")
for row in rows:
    print(row)
for count in counts:
    print(count)

In [ ]:
from pipeline.embedder import delete_collection

delete_collection(session_id)
print(session_id, "deleted")